In [1]:
# NLP303 Assessment 3
#
# Detecting AI-generated Text
# Using Transformer-Based Classification
#
#
#
# Tibor Titusz Tarcsai - A00121308
#
# Jonathan Lim - A00142089
#
# Thomas Galindo Salazar - A00129258
#
#
#
# The purpose of this implementation is to demonstrate the building of a working protoype for a Classification task based on the Assessment 2 proposal.
#
#
#***********************
# HOW TO RUN THE CODE:
#***********************
#
# 1. - The notebook can run either in Jupyter Notebook or Google Colab 
#    - If using Google Colab, a Google Drive account is required for data access and storage.
#      Link for running on Colab: 
#
# 2. All helper functions are imported and loaded from the src folder
#
# 3. In Section [2], the "find_cat_dog_lion_tiger_folders" path finding function automatically locates the dataset folder,
#    so NO manual path configuration is required from the user.

In [1]:
# Environment Setup and Dependencies

In [1]:
!pip install -r ../requirements.txt
print("### Dependencies installed successfully ###")

### Dependencies installed successfully ###


In [2]:
# Initial Library Imports
# --------------------------------
# - torch for ... implementation
#
#
#
#
#

import os
import torch
import pandas as pd
import transformers


from torchinfo import summary
from transformers import pipeline

import sys
sys.path.append("../")


from src.downloader import DatasetDownloader

from src.datasetbuilder import DatasetBuilder

from src.preprocessing import TextPreprocessor

from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


# Section 1 - Load Dataset

In [4]:
# Initialises the downloader class
downloader = DatasetDownloader()

# Downloads raw datasets from google drive
print("### Downloading Dataset from Google Drive... ###")
downloader.start_downloading_dataset()

### Downloading Dataset from Google Drive... ###


Downloading...
From: https://drive.google.com/uc?id=1v8ZKV3p6KLDMsOscVLj1Z5zgNYGJfaQp
To: /home/titus/projects/roberta-ai-text-detector/data/humanised_v2_first_2400.csv
100%|██████████| 7.48M/7.48M [00:01<00:00, 5.51MB/s]
Downloading...
From: https://drive.google.com/uc?id=1U9Lhpo2qet7dPAuswxHFGAJEggc26bR0
To: /home/titus/projects/roberta-ai-text-detector/data/ai_polished_v2_first_2400.csv
100%|██████████| 7.36M/7.36M [00:01<00:00, 5.51MB/s]
Downloading...
From: https://drive.google.com/uc?id=182-e58HGw67tacTudS7wacm6DZZCbMEZ
To: /home/titus/projects/roberta-ai-text-detector/data/pure_ai_v2_first_2400.csv
100%|██████████| 4.85M/4.85M [00:00<00:00, 5.81MB/s]
Downloading...
From: https://drive.google.com/uc?id=15BDFQcaylNmK6Uy-BaKKe__jXQ6MgdmK
To: /home/titus/projects/roberta-ai-text-detector/data/pure_human_v2_first_2400.csv
100%|██████████| 3.96M/3.96M [00:00<00:00, 5.90MB/s]


In [6]:
# Restructures datasets


raw_dataset_paths = [
            {
                "pure_human": "../data/pure_human_v2_first_2400.csv"
            },
            {
                "pure_ai": "../data/pure_ai_v2_first_2400.csv"
            },
            {
                "ai_polished": "../data/ai_polished_v2_first_2400.csv"
            },
            {
                "humanised": "../data/humanised_v2_first_2400.csv"
            },
        ]


builder = DatasetBuilder()

raw_dataset = builder.build_dataset(
    raw_dataset_paths[0]["pure_human"],     # Label 0
    raw_dataset_paths[1]["pure_ai"],        # Label 1
    raw_dataset_paths[2]["ai_polished"],    # Label 2
    raw_dataset_paths[3]["humanised"],      # Label 3
)


In [7]:
print(len(raw_dataset))

9600


In [8]:
raw_dataset.iloc[2458].text

"http://en.wikipedia.org/wiki/Kathy_Arendsen\n\nKathy Arendsen (born January 2, 1956) is a Minnesota lawyer, farmer, and politician who has been a Minnesota Senator, representing the District 13, and she is currently a commissioner of the Minnesota Public Utilities Commission (PUC).\n\nArendsen is a member of the Republican Party of Minnesota.  Elected in 2008, Arendsen replaced Mark B. Neumann in the state's legislature. She lost to Dave Senjem when she challenged him in the 2012 Republican primary for the U.S. Senate. She was elected to the PUC in 2018.\n\nArendsen is a former two-term president of the Minnesota Senate, serving from 2009 to 2011. She is the first woman to serve as President of the Minnesota Senate.\n\nArendsen graduated from Blooming Prairie High School, and from Northfield High School in 1974. She earned a Bachelor of Science in Agriculture from Minnesota State University-Moorhead. She attended graduate school at Concordia Language Villages, where she was introduced

In [12]:
# Selects rows 
raw_dataset.iloc[7200:7250]


,text,label,label_name,cleaned_text
7200,"A recent paper titled ""Future-AI: Guiding Prin...",3,ai_written_humanised,"A recent paper titled ""Future-AI: Guiding Prin..."
7201,The phenomenon of matrix singularity has garne...,3,ai_written_humanised,The phenomenon of matrix singularity has garne...
7202,Protecting secret messages sent over anonymous...,3,ai_written_humanised,Protecting secret messages sent over anonymous...
7203,The phenomenon of sputtering on polar surfaces...,3,ai_written_humanised,The phenomenon of sputtering on polar surfaces...
7204,\nThe introduction of two new scheduling primi...,3,ai_written_humanised,The introduction of two new scheduling primit...
7205,"""A quantum mechanical description of anisotrop...",3,ai_written_humanised,"""A quantum mechanical description of anisotrop..."
7206,The development of a self-perfect absorber has...,3,ai_written_humanised,The development of a self-perfect absorber has...
7207,Lagrangian manifolds in Hilbert space have bee...,3,ai_written_humanised,Lagrangian manifolds in Hilbert space have bee...
7208,Researchers investigated the upper limits on g...,3,ai_written_humanised,Researchers investigated the upper limits on g...
7209,The article examines the supersymmetric topolo...,3,ai_written_humanised,The article examines the supersymmetric topolo...


# Section 2 - Text Preprocessing

In [10]:
# Drops rows where 'text' is missing
raw_dataset = raw_dataset.dropna(subset=["text"]).reset_index(drop=True)

print(len(raw_dataset))

9600


In [11]:
text_preprocessor = TextPreprocessor()

raw_dataset["cleaned_text"] = raw_dataset["text"].apply(text_preprocessor.clean_text)



print(raw_dataset[["cleaned_text", "text"]].head(2))

                                        cleaned_text  \
0  For the first time in nearly 3 years i can fin...   
1  Donnie Yen is a long time favorite of mine, al...   

                                                text  
0  For the first time in nearly 3 years i can fin...  
1  Donnie Yen is a long time favorite of mine, al...  


In [ ]:
# Section 3 - Tokenisation (BPE)

In [ ]:
# Section 4 - Load Pre-trained RoBERTa

In [ ]:
# Section 5- Fine-tuning

In [ ]:
# Section 6 - Model Inference

In [ ]:
# Section 7 - Threshold Classification

In [ ]:
# Section 8 - Evaluation

In [ ]:
# Section 9 - Visualisations

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

model = AutoModelForSequenceClassification.from_pretrained(
    "fakespot-ai/roberta-base-ai-text-detection-v1"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# Prints model summary
summary(model, input_size=(1, 512), dtypes=[torch.long])

Layer (type:depth-idx)                                       Output Shape              Param #
RobertaForSequenceClassification                             [1, 2]                    --
├─RobertaModel: 1-1                                          [1, 512, 768]             --
│    └─RobertaEmbeddings: 2-1                                [1, 512, 768]             --
│    │    └─Embedding: 3-1                                   [1, 512, 768]             38,603,520
│    │    └─Embedding: 3-2                                   [1, 512, 768]             768
│    │    └─Embedding: 3-3                                   [1, 512, 768]             394,752
│    │    └─LayerNorm: 3-4                                   [1, 512, 768]             1,536
│    │    └─Dropout: 3-5                                     [1, 512, 768]             --
│    └─RobertaEncoder: 2-2                                   [1, 512, 768]             --
│    │    └─ModuleList: 3-6                                  --               

In [ ]:
df = pd.read_csv("data/HUMAN_written_then_AI_polished.csv")

In [ ]:
print(df.head(200))

                                      id  \
0   e5e058ce-be2b-459d-af36-32532aaba5ff   
1   f95b107b-d176-4af5-90f7-4d0bb20caf93   
2   856d8972-9e3d-4544-babc-0fe16f21e04d   
3   fbc8a5ea-90fa-47b8-8fa7-73dd954f1524   
4   72c41b8d-0069-4886-b734-a4000ffca286   
5   72fe360b-cce6-4daf-b66a-1d778f5964f8   
6   df594cf4-9a0c-4488-bcb3-68f41e2d5a16   
7   853c0e51-7dd5-4bb5-8286-e4aa8820173b   
8   1649f195-8f98-4c79-92b6-54a5ca9261fa   
9   5e23ab14-b85f-48e8-9aa3-15452e73524e   
10  ddcb207c-a790-4e16-a053-4aced58d7c15   
11  b00bf7dc-4de9-4ab4-9962-a16e0b5f4628   
12  04d3809c-0abe-4bee-b1d2-9787af95362f   
13  06bffeb2-bea0-4b0b-b60d-767ba9b660a7   
14  5eb88a59-eb5a-49ea-8304-f67efe338921   
15  1389aa64-25fb-4e56-9358-ef34143bfea9   
16  d0064195-c22e-4550-a265-6b372deea3e0   
17  417afaa2-2d21-4df1-953b-768647de9980   
18  ce898c28-428f-446f-975e-a1265942f2da   
19  380cd71d-3300-422c-9cde-8a63e71f2797   
20  c093400c-2bd2-4e0d-a732-f99d499d58a9   
21  05f40b6d-67cf-4a6e-ad2f-cfe0